In [19]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)
from pyspark.sql.functions import (
    col, when, lit, to_date, hour, dayofweek, month,
    count, avg, sum as _sum, isnull, trim, expr
)
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml import Pipeline
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark import StorageLevel

spark = (
    SparkSession.builder
    .appName("s04-regresion-retrasos-leonardo")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

ORIGEN_DATOS = "/opt/s04-ml-distribuido-regresion/data"
ARTIFACTS = "/opt/s04-ml-distribuido-regresion/artifacts"

print("✅ Spark listo")

✅ Spark listo


## Fase 1 — Business Understanding

**Objetivo:** Estimar `dep_delay` (minutos de retraso en salida) a partir de variables conocidas antes del despegue.

**Alcance:** Comparar LinearRegression (3 configs) vs. RandomForestRegressor. Reportar RMSE, R² y MAE. Guardar el ganador.

**Decisión que habilita:** Ajustar buffers de tiempo en rutas problemáticas.

In [20]:
schema_vuelos = StructType([
    StructField("year", IntegerType(), True),
    StructField("month", IntegerType(), True),
    StructField("day_of_month", IntegerType(), True),
    StructField("day_of_week", IntegerType(), True),
    StructField("fl_date", StringType(), True),
    StructField("op_unique_carrier", StringType(), True),
    StructField("op_carrier_fl_num", IntegerType(), True),
    StructField("origin", StringType(), True),
    StructField("origin_city_name", StringType(), True),
    StructField("origin_state_nm", StringType(), True),
    StructField("dest", StringType(), True),
    StructField("dest_city_name", StringType(), True),
    StructField("dest_state_nm", StringType(), True),
    StructField("crs_dep_time", StringType(), True),
    StructField("dep_time", StringType(), True),
    StructField("dep_delay", DoubleType(), True),
    StructField("taxi_out", DoubleType(), True),
    StructField("wheels_off", StringType(), True),
    StructField("wheels_on", StringType(), True),
    StructField("taxi_in", DoubleType(), True),
    StructField("crs_arr_time", StringType(), True),
    StructField("arr_time", StringType(), True),
    StructField("arr_delay", DoubleType(), True),
    StructField("cancelled", DoubleType(), True),
    StructField("cancellation_code", StringType(), True),
    StructField("diverted", DoubleType(), True),
    StructField("crs_elapsed_time", DoubleType(), True),
    StructField("actual_elapsed_time", DoubleType(), True),
    StructField("air_time", DoubleType(), True),
    StructField("distance", DoubleType(), True),
    StructField("carrier_delay", DoubleType(), True),
    StructField("weather_delay", DoubleType(), True),
    StructField("nas_delay", DoubleType(), True),
    StructField("security_delay", DoubleType(), True),
    StructField("late_aircraft_delay", DoubleType(), True),
])

df = spark.read.csv(
    f"{ORIGEN_DATOS}/flight_data_2024.csv",
    header=True,
    schema=schema_vuelos,
)

print(f"Total de filas: {df.count():,}")
df.printSchema()

[Stage 200:=================>                                     (5 + 11) / 16]

Total de filas: 7,079,081
root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day_of_month: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- fl_date: string (nullable = true)
 |-- op_unique_carrier: string (nullable = true)
 |-- op_carrier_fl_num: integer (nullable = true)
 |-- origin: string (nullable = true)
 |-- origin_city_name: string (nullable = true)
 |-- origin_state_nm: string (nullable = true)
 |-- dest: string (nullable = true)
 |-- dest_city_name: string (nullable = true)
 |-- dest_state_nm: string (nullable = true)
 |-- crs_dep_time: string (nullable = true)
 |-- dep_time: string (nullable = true)
 |-- dep_delay: double (nullable = true)
 |-- taxi_out: double (nullable = true)
 |-- wheels_off: string (nullable = true)
 |-- wheels_on: string (nullable = true)
 |-- taxi_in: double (nullable = true)
 |-- crs_arr_time: string (nullable = true)
 |-- arr_time: string (nullable = true)
 |-- arr_delay: double (nullable = t

In [33]:
df.select("dep_delay", "distance", "crs_elapsed_time").describe().show()

df.groupBy("month").count().orderBy("month").show()

+-------+-----------------+-----------------+-----------------+
|summary|        dep_delay|         distance| crs_elapsed_time|
+-------+-----------------+-----------------+-----------------+
|  count|          6986111|          7079081|          7079080|
|   mean|12.67708199884027|833.9061503887299|146.7664685241585|
| stddev|56.05997027106313|596.2535939770956|72.38692355437206|
|    min|            -96.0|             11.0|           -160.0|
|    max|           3777.0|           5095.0|           1326.0|
+-------+-----------------+-----------------+-----------------+



[Stage 388:================================================>      (14 + 2) / 16]

+-----+------+
|month| count|
+-----+------+
|    1|547271|
|    2|519221|
|    3|591767|
|    4|582205|
|    5|609743|
|    6|611132|
|    7|634613|
|    8|619025|
|    9|582622|
|   10|615497|
|   11|575404|
|   12|590581|
+-----+------+



In [34]:
# Reducimos a 2 meses para no saturar la memoria
df_q1 = df.filter((col("month") >= 1) & (col("month") <= 2))

print(f"Filas del primer bimestre 2024: {df_q1.count():,}")

[Stage 391:===>                                                   (1 + 15) / 16]

Filas del primer bimestre 2024: 1,066,492


In [35]:
df_prep = (
    df_q1
    .withColumn("fecha", to_date(col("fl_date"), "yyyy-MM-dd"))
    .withColumn(
        "crs_dep_hour",
        (col("crs_dep_time").cast("int") / 100).cast("int")
    )
    .withColumn("dow", dayofweek(col("fecha")))
)

df_prep.select(
    "fecha", "crs_dep_time", "crs_dep_hour",
    "dow", "month", "op_unique_carrier", "origin", "dest",
    "distance", "dep_delay"
).show(5, truncate=False)

+----------+------------+------------+---+-----+-----------------+------+----+--------+---------+
|fecha     |crs_dep_time|crs_dep_hour|dow|month|op_unique_carrier|origin|dest|distance|dep_delay|
+----------+------------+------------+---+-----+-----------------+------+----+--------+---------+
|2024-01-01|1252        |12          |2  |1    |9E               |JFK   |DTW |509.0   |-5.0     |
|2024-01-01|1015        |10          |2  |1    |9E               |MSP   |CLE |622.0   |-14.0    |
|2024-01-01|1415        |14          |2  |1    |9E               |JFK   |RIC |288.0   |-4.0     |
|2024-01-01|1650        |16          |2  |1    |9E               |RIC   |JFK |288.0   |-7.0     |
|2024-01-01|1015        |10          |2  |1    |9E               |DTW   |MKE |237.0   |-5.0     |
+----------+------------+------------+---+-----+-----------------+------+----+--------+---------+
only showing top 5 rows


In [36]:
df_clean = df_prep.filter(
    (col("cancelled") == 0) &
    (col("dep_delay").isNotNull()) &
    (col("distance").isNotNull()) &
    (col("crs_elapsed_time").isNotNull())
)

print(f"Filas antes: {df_prep.count():,}")
print(f"Filas después: {df_clean.count():,}")
print(f"Eliminadas: {df_prep.count() - df_clean.count():,}")

Filas antes: 1,066,492


Filas después: 1,043,101


[Stage 404:===>                                                   (1 + 15) / 16]

Eliminadas: 23,391


In [37]:
total = df_clean.count()
sin_dup = df_clean.dropDuplicates([
    "fl_date", "op_unique_carrier", "op_carrier_fl_num",
    "origin", "dest", "crs_dep_time"
]).count()

print(f"Total: {total:,}")
print(f"Sin duplicados: {sin_dup:,}")
print(f"Duplicados: {total - sin_dup:,}")

df_clean = df_clean.dropDuplicates([
    "fl_date", "op_unique_carrier", "op_carrier_fl_num",
    "origin", "dest", "crs_dep_time"
])

[Stage 410:================================================>      (14 + 2) / 16]

Total: 1,043,101
Sin duplicados: 1,042,839
Duplicados: 262


In [38]:
df_clean.write.mode("overwrite") \
    .partitionBy("month") \
    .parquet(f"{ARTIFACTS}/vuelos_silver")
print("✅ Silver guardado")

✅ Silver guardado


In [49]:
# Leer de vuelta la salida particionada
df_verificacion = spark.read.parquet(f"{ARTIFACTS}/vuelos_silver")

print(f"Filas leídas: {df_verificacion.count():,}")
print(f"Columnas: {df_verificacion.columns}")
df_verificacion.printSchema()

# Verificar que no se perdió nada (ida y vuelta)
assert df_verificacion.count() == df_clean.count(), "¡Se perdieron filas en la escritura!"
print("✅ Verificación de ida y vuelta OK")

Filas leídas: 1,042,839
Columnas: ['year', 'day_of_month', 'day_of_week', 'fl_date', 'op_unique_carrier', 'op_carrier_fl_num', 'origin', 'origin_city_name', 'origin_state_nm', 'dest', 'dest_city_name', 'dest_state_nm', 'crs_dep_time', 'dep_time', 'dep_delay', 'taxi_out', 'wheels_off', 'wheels_on', 'taxi_in', 'crs_arr_time', 'arr_time', 'arr_delay', 'cancelled', 'cancellation_code', 'diverted', 'crs_elapsed_time', 'actual_elapsed_time', 'air_time', 'distance', 'carrier_delay', 'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay', 'fecha', 'crs_dep_hour', 'dow', 'month']
root
 |-- year: integer (nullable = true)
 |-- day_of_month: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- fl_date: string (nullable = true)
 |-- op_unique_carrier: string (nullable = true)
 |-- op_carrier_fl_num: integer (nullable = true)
 |-- origin: string (nullable = true)
 |-- origin_city_name: string (nullable = true)
 |-- origin_state_nm: string (nullable = true)
 |-- 

[Stage 604:=====================>                                   (3 + 5) / 8]

✅ Verificación de ida y vuelta OK


In [50]:
# Verificar que Spark usa PartitionFilters al filtrar por month
df_verificacion.filter(col("month") == 1).explain(True)

== Parsed Logical Plan ==
'Filter '`=`('month, 1)
+- Relation [year#12075,day_of_month#12076,day_of_week#12077,fl_date#12078,op_unique_carrier#12079,op_carrier_fl_num#12080,origin#12081,origin_city_name#12082,origin_state_nm#12083,dest#12084,dest_city_name#12085,dest_state_nm#12086,crs_dep_time#12087,dep_time#12088,dep_delay#12089,taxi_out#12090,wheels_off#12091,wheels_on#12092,taxi_in#12093,crs_arr_time#12094,arr_time#12095,arr_delay#12096,cancelled#12097,cancellation_code#12098,diverted#12099,... 13 more fields] parquet

== Analyzed Logical Plan ==
year: int, day_of_month: int, day_of_week: int, fl_date: string, op_unique_carrier: string, op_carrier_fl_num: int, origin: string, origin_city_name: string, origin_state_nm: string, dest: string, dest_city_name: string, dest_state_nm: string, crs_dep_time: string, dep_time: string, dep_delay: double, taxi_out: double, wheels_off: string, wheels_on: string, taxi_in: double, crs_arr_time: string, arr_time: string, arr_delay: double, cancell

In [39]:
categoricas = ["op_unique_carrier", "origin", "dest"]
numericas = ["crs_dep_hour", "dow", "distance", "crs_elapsed_time", "month"]

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="skip")
    for c in categoricas
]

encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_oh")
    for c in categoricas
]

predictores = numericas + [f"{c}_oh" for c in categoricas]
assembler = VectorAssembler(inputCols=predictores, outputCol="features")

df_ml = df_clean.select(numericas + categoricas + ["dep_delay"])

print("✅ Pipeline ML preparado")

✅ Pipeline ML preparado


In [40]:
# Aplicar pipeline de preparación
pipeline_prep = Pipeline(stages=indexers + encoders + [assembler])
df_ml_prep = (
    pipeline_prep.fit(df_ml)
    .transform(df_ml)
    .select("features", "dep_delay")
)

# Persistir con MEMORY_AND_DISK (si no cabe en RAM, va a disco sin fallar)
df_ml_prep.persist(StorageLevel.MEMORY_AND_DISK)
print(f"Total preparado: {df_ml_prep.count():,}")

# Split 80/20
df_train, df_test = df_ml_prep.randomSplit([0.8, 0.2], seed=42)

# Persistir train y test por separado
df_train.persist(StorageLevel.MEMORY_AND_DISK)
df_test.persist(StorageLevel.MEMORY_AND_DISK)

print(f"Entrenamiento: {df_train.count():,}")
print(f"Prueba: {df_test.count():,}")

Total preparado: 1,042,839


Entrenamiento: 834,588


[Stage 465:=================================================>       (7 + 1) / 8]

Prueba: 208,251


In [41]:
lr_base = LinearRegression(featuresCol="features", labelCol="dep_delay")
modelo_base = lr_base.fit(df_train)
pred_base = modelo_base.transform(df_test)

print("Coeficientes:", modelo_base.coefficients[:5])
print("Intercepto:", modelo_base.intercept)

26/09/11 17:34:11 WARN Instrumentation: [8a07545b] regParam is zero, which might cause numerical instability and overfitting.
                                                                                

Coeficientes: [ 8.33789112e-01 -4.71478759e-01 -5.49029139e-03  4.90873816e-02
 -8.16029175e+00]
Intercepto: 17.201255073858206


In [42]:
def evaluar(predicciones, nombre):
    resultados = {}
    for metrica in ["rmse", "r2", "mae"]:
        evaluador = RegressionEvaluator(
            labelCol="dep_delay",
            predictionCol="prediction",
            metricName=metrica
        )
        resultados[metrica.upper()] = evaluador.evaluate(predicciones)
    print(f"{nombre}: RMSE={resultados['RMSE']:.4f} R2={resultados['R2']:.4f} MAE={resultados['MAE']:.4f}")
    return resultados

resultados_base = evaluar(pred_base, "LinearRegression base")

LinearRegression base: RMSE=56.4575 R2=0.0176 MAE=22.9645


In [43]:
configuraciones = [
    {"nombre": "Sin regularización", "regParam": 0.0, "elasticNetParam": 0.0},
    {"nombre": "Ridge (L2)", "regParam": 0.1, "elasticNetParam": 0.0},
    {"nombre": "Elastic Net (L1+L2)", "regParam": 0.1, "elasticNetParam": 0.5},
]

comparacion = []
for cfg in configuraciones:
    lr = LinearRegression(
        featuresCol="features", labelCol="dep_delay",
        regParam=cfg["regParam"], elasticNetParam=cfg["elasticNetParam"]
    )
    modelo = lr.fit(df_train)
    pred = modelo.transform(df_test)
    r = evaluar(pred, cfg["nombre"])
    r["Configuración"] = cfg["nombre"]
    comparacion.append(r)

import pandas as pd
pd.DataFrame(comparacion)[["Configuración", "RMSE", "R2", "MAE"]]

26/09/11 17:34:31 WARN Instrumentation: [744df216] regParam is zero, which might cause numerical instability and overfitting.
                                                                                

Sin regularización: RMSE=56.4575 R2=0.0176 MAE=22.9645


Ridge (L2): RMSE=56.4582 R2=0.0176 MAE=22.9649


Elastic Net (L1+L2): RMSE=56.4580 R2=0.0176 MAE=22.9573


,Configuración,RMSE,R2,MAE
0,Sin regularización,56.457544,0.017581,22.964520
1,Ridge (L2),56.458176,0.017559,22.964851
2,Elastic Net (L1+L2),56.458014,0.017565,22.957327


In [44]:
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="dep_delay",
    numTrees=30,              # reducido de 50
    maxDepth=6,               # reducido de 8
    minInstancesPerNode=5,    # evita hojas con pocos datos
    seed=42
)
modelo_rf = rf.fit(df_train)
pred_rf = modelo_rf.transform(df_test)

resultados_rf = evaluar(pred_rf, "Random Forest")

26/09/11 17:34:51 WARN MemoryStore: Not enough space to cache rdd_1638_5 in memory! (computed 233.4 MiB so far)
26/09/11 17:34:51 WARN MemoryStore: Not enough space to cache rdd_1638_0 in memory! (computed 233.4 MiB so far)
26/09/11 17:34:51 WARN MemoryStore: Not enough space to cache rdd_1638_6 in memory! (computed 233.4 MiB so far)
26/09/11 17:34:51 WARN MemoryStore: Not enough space to cache rdd_1638_1 in memory! (computed 233.4 MiB so far)
26/09/11 17:34:51 WARN MemoryStore: Not enough space to cache rdd_1638_2 in memory! (computed 233.4 MiB so far)
26/09/11 17:34:51 WARN BlockManager: Persisting block rdd_1638_0 to disk instead.
26/09/11 17:34:51 WARN BlockManager: Persisting block rdd_1638_1 to disk instead.
26/09/11 17:34:51 WARN BlockManager: Persisting block rdd_1638_6 to disk instead.
26/09/11 17:34:51 WARN BlockManager: Persisting block rdd_1638_2 to disk instead.
26/09/11 17:34:51 WARN BlockManager: Persisting block rdd_1638_5 to disk instead.
26/09/11 17:34:54 WARN MemoryS

Random Forest: RMSE=56.4404 R2=0.0182 MAE=22.9204


In [48]:
importancias_array = modelo_rf.featureImportances.toArray()
print(f"Total features: {len(importancias_array)}")
print(f"Suma: {importancias_array.sum():.4f}")
print()

# Top 15 por índice (sin mapear a nombres, honesto)
top_indices = importancias_array.argsort()[-15:][::-1]
print("Top 15 features por importancia:")
for rank, idx in enumerate(top_indices, 1):
    print(f"  {rank:2d}. Feature[{idx:3d}] = {importancias_array[idx]:.4f}")

Total features: 685
Suma: 1.0000

Top 15 features por importancia:
   1. Feature[  4] = 0.2648
   2. Feature[  0] = 0.2607
   3. Feature[  1] = 0.1003
   4. Feature[  6] = 0.0482
   5. Feature[ 10] = 0.0400
   6. Feature[367] = 0.0319
   7. Feature[  9] = 0.0179
   8. Feature[ 34] = 0.0157
   9. Feature[  7] = 0.0155
  10. Feature[ 13] = 0.0142
  11. Feature[353] = 0.0129
  12. Feature[  2] = 0.0129
  13. Feature[  3] = 0.0124
  14. Feature[  5] = 0.0112
  15. Feature[117] = 0.0088


In [46]:
comparacion_final = comparacion + [{"Configuración": "Random Forest", **resultados_rf}]
pd.DataFrame(comparacion_final)[["Configuración", "RMSE", "R2", "MAE"]]

,Configuración,RMSE,R2,MAE
0,Sin regularización,56.457544,0.017581,22.964520
1,Ridge (L2),56.458176,0.017559,22.964851
2,Elastic Net (L1+L2),56.458014,0.017565,22.957327
3,Random Forest,56.440443,0.018176,22.920391


In [47]:
df_comp = pd.DataFrame(comparacion_final)
ganador_nombre = df_comp.loc[df_comp["RMSE"].idxmin(), "Configuración"]
print(f"🏆 Ganador: {ganador_nombre}")

modelo_ganador = modelo_rf  # ajusta según el ganador real
modelo_ganador.write().overwrite().save(f"{ARTIFACTS}/modelo_retrasos_rf")
print(f"✅ Modelo guardado en {ARTIFACTS}/modelo_retrasos_rf")

🏆 Ganador: Random Forest


✅ Modelo guardado en /opt/s04-ml-distribuido-regresion/artifacts/modelo_retrasos_rf


## Fase 6 — Cierre

**Reflexión técnica:**

Random Forest superó al modelo lineal porque la relación entre predictores y `dep_delay` no es lineal. La regularización no mejoró el modelo lineal, lo que sugiere que no había overfitting.

**¿Por qué VectorAssembler es obligatorio?**
Spark MLlib requiere que todos los predictores estén en una sola columna vectorial por fila.

**¿Por qué varias métricas?**
RMSE penaliza errores grandes, MAE da el error promedio, R² indica la proporción de varianza explicada. Una sola métrica puede ocultar problemas.